# 05 · Discrete-time fully-differential amplifier: R-NSGA-II optimization with novaad + ngspice in the loop

```
                 VDD ─────────┬───────────────────┬──────────
                              │                   │
          cmfb ──────────┤ M3            M4 ├──────── cmfb
                              │                   │
              outn ───────────┼──┬─| |─-┬─-| |─┬──┼─────────── outp
                              │  │ C_CM │ C_CM │  │
                              │  │      cmfb   │  │
              inp ───────┤ M1 │  CL           CL  │ M2 ├───── inn
                              └────────┬──────────┘
                                     tail
   IREF ↓                              │
        nb ──┬──────────────────┤ M5  (active tail current source)
       M6 ├──┘ (diode)                 │
            GND ──────────────────────GND
```

IHP SG13G2 1.2 V core devices (`sg13_lv_nmos` / `sg13_lv_pmos`), `VDD = 1.2 V`.

* **M1/M2**: NMOS differential pair.
* **M3/M4**: PMOS active loads. Their gates sit at `cmfb`, which two capacitors `C_CM` couple to both outputs. For the differential signal `cmfb` is a virtual ground, so the loads act as current sources and the gain is `gm1·(ro1 ‖ ro3)`.
* **Switched-capacitor CM sensing**: a transmission gate (`MSN`/`MSP`, minimum length, with half-width dummies on both sides to cancel charge injection) resets `cmfb` to the reference `avcm` during the clock phase `ph`. During `phb` it opens: `cmfb` floats and follows the output common mode through `C_CM`. `ph`/`phb` are non-overlapping, anti-phase 50 MHz clocks.
* **M5/M6**: an NMOS current mirror, the *active source* of the tail current `I_TAIL`.

For each candidate design, **novaad** sizes every transistor from gm/id, L and current, and **ngspice** measures it (OP, AC, noise) in the **static reset phase**. The clocked behaviour of the picked design is verified in section 6 with a periodic-steady-state Bode plot and transients. **pymoo's R-NSGA-II** searches the trade-off between four objectives:

| objective | definition |
|---|---|
| minimize **spec error** | RMS of the relative errors: `measured / target − 1` for each `SPEC_TARGETS` entry (gain, GBW, output CM, power dissipation), and the excess `max(0, measured / limit − 1)` for each `SPEC_LIMITS` entry (input-referred noise) |
| minimize **area** | `Σ W·L` of M1–M6 and the reset switches + MIM area of the two sensing capacitors `2·C_CM/CAP_DENSITY` |
| minimize **power** | `VDD·I(VDD)` |
| maximize **Schreier FoM** | `FoM_S = SNR_dB + 10·log10(GBW / P)` |

`SNR = (A²/2) / Vn²`, where:
- `A` is the peak differential output swing allowed by the headroom of M1 and M3 at the simulated operating point.
- `Vn²` is the differential output noise integrated from `NOISE_FMIN` to `GBW`.

Maximizing `FoM_S` rewards both SNR and GBW per watt.

The specs are **targets**, not bounds: every measured parameter should land as close as possible to its target, so the combined spec error is minimized together with area, power and −FoM. There are **no hard constraints**: the result is a ranked list of the `N_CLOSEST` designs closest to the targets, and every design reports its saturation margin so devices that leave saturation are visible.

**R-NSGA-II** (reference-point NSGA-II) is NSGA-II with designer preferences. Instead of spreading solutions over the whole Pareto front, it concentrates them around **reference points**: aspiration values for `(spec error, area, power, −FoM)` set in `REF_POINTS`. `EPSILON` controls how tightly they cluster.

The Pareto front is plotted **live while the optimizer runs**:
- R-NSGA-II runs in a background thread.
- A second thread pushes the current front, with the reference points marked, into a Bokeh figure.

Run `01_pdk_ngspice_lut_extraction.ipynb` first to create the LUTs.


In [1]:
# ── Operating conditions: edit these ────────────────────────────────────────
VDD = 1.2              # supply [V]: IHP SG13G2 1.2 V core devices
VIN_CM = 0.6           # input common mode [V]
CL = 0.5e-12           # load capacitance per output [F]
NOISE_FMIN = 1.0       # lower bound of the noise integration [Hz] (upper bound = GBW)

# ── Clocks: non-overlapping anti-phase ph (reset) / phb (amplify) ───────────
F_CLK = 50e6           # clock frequency [Hz]
T_NOV = 0.1e-9         # gap between ph falling and phb rising (and back) [s]
T_EDGE = 0.5e-9        # clock rise/fall time [s]

# ── CM reset switch: transmission gate with half-width dummies on both sides ─
W_SWITCH_N = 1e-6      # NMOS switch width [m]
W_SWITCH_P = 2e-6      # PMOS switch width [m]
L_SWITCH = 0.13e-6     # minimum length [m]

# ── Spec targets: the optimizer drives each measured parameter toward its target ──
SPEC_TARGETS = {
    "gain_db": 26,        # differential DC gain [dB]
    "gbw": 500e6,         # differential gain-bandwidth product A0·f-3dB [Hz]
    "vocm": 0.6,          # output common mode [V]
    "power": 500e-6,      # power dissipation VDD·I(VDD) [W], including the IREF branch
}
# one-sided limits: only exceeding the limit counts as error (noise can't usefully be pushed *up* to a target)
SPEC_LIMITS = {
    "vn_in": 60e-6,       # input-referred noise [Vrms]: output noise (NOISE_FMIN..GBW) / DC gain
}

# ── Area model ──────────────────────────────────────────────────────────────
CAP_DENSITY = 1.5e-3   # MIM capacitance density [F/m²] (cmim, 1.5 fF/um²)

# ── Design space (bounds of the 8 optimization variables) ───────────────────
GMID_IN_BOUNDS = (14, 24)        # input pair gm/id [1/V]
GMID_LOAD_BOUNDS = (6, 16)       # load gm/id [1/V]
GMID_TAIL_BOUNDS = (6, 16)       # tail gm/id [1/V]
I_TAIL_BOUNDS = (10e-6, 500e-6)  # tail current [A]
C_CM_BOUNDS = (10e-15, 500e-15)    # CM sensing capacitors [F]
# channel lengths are picked from the LUT grid (L_IN, L_LOAD, L_TAIL)

# ── R-NSGA-II ───────────────────────────────────────────────────────────────
REF_POINTS = [         # aspiration points (spec error [%], area [um²], power [uW], -FoM_S [dB])
    (0, 600, 400, -172),    # on all targets, moderate area
    (0, 250, 300, -170),    # on all targets, compact and lower power
]
EPSILON = 0.05         # spread of the solutions around each reference point (normalized objectives)
POP_SIZE = 48
N_GEN = 40
SEED = 1
N_WORKERS = 8          # parallel ngspice evaluations
PLOT_PERIOD = 2.0      # seconds between live plot refreshes
N_CLOSEST = 10         # length of the final list of designs closest to the targets

# ── Verification of the picked design (section 6) ───────────────────────────
TRAN_AMP = 1e-3        # differential sine amplitude [V]
# ramp and pulse levels of the differential input vd: with VIN_CM = VDD/2, vd = ±VDD swings
# each input (VIN_CM ± vd/2) over the full supply
RAMP_LOW = -VDD        # differential ramp start [V]
RAMP_HIGH = VDD        # differential ramp end [V]
STEP_LOW = -VDD        # differential pulse low level [V]
STEP_HIGH = VDD        # differential pulse high level [V]
# time scales follow the design's own -3 dB corner f3, with tau = 1 / (2*pi*f3)
TRAN_FREQ_RATIO = 0.1  # sine frequency = ratio * f3
RAMP_TAUS = 200        # ramp duration in tau: slow enough to trace the static transfer curve
STEP_TAUS = 20         # pulse window in tau: vd steps up at 10 % and back down at 50 %
STEPS_PER_CLK = 100    # transient time step: 1 / (F_CLK·STEPS_PER_CLK)

# ── Periodic steady state (swept sine) of the clocked amplifier ─────────────
PSS_FMIN = F_CLK / 500       # lowest test frequency [Hz]
PSS_FMAX = 2 * F_CLK         # highest test frequency [Hz]: past Nyquist, to show the aliasing region
PSS_POINTS = 24              # log-spaced test frequencies
PSS_AMP = 1e-3               # differential test amplitude [V]
PSS_MAX_CYCLES = 500         # longest common period of input and clock, in clock cycles
PSS_SETTLE_CYCLES = 20       # minimum settling before the analysed period, in clock cycles

NCH_LUT = "luts/sg13_lv_nch.csv"
PCH_LUT = "luts/sg13_lv_pch.csv"
REF_WIDTH = 10e-6


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# the live plot embeds a srcdoc <iframe> on purpose (IFrame needs a URL)
warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")

import shutil
from fractions import Fraction
from math import ceil, log10, pi
from multiprocessing.pool import ThreadPool
from threading import Event, Lock, Thread

import numpy as np
from html import escape

from IPython.display import HTML, display
from pandas import DataFrame
from bokeh.embed import file_html
from bokeh.io import output_notebook, show
from bokeh.layouts import gridplot, row
from bokeh.models import ColorBar, ColumnDataSource, LinearColorMapper, Span
from bokeh.palettes import Viridis256
from bokeh.plotting import figure
from bokeh.resources import INLINE
from pymoo.algorithms.moo.rnsga2 import RNSGA2
from pymoo.core.callback import Callback
from pymoo.core.problem import ElementwiseProblem
from pymoo.parallelization import StarmapParallelization
from pymoo.optimize import minimize

import spice
from novaad import Device, DeviceSizingSpecification as Spec
from utils import dc_tables, dc_vectors, footprint, tran_plot

output_notebook(hide_banner=True)
nch = Device(NCH_LUT, device_type="nch", ref_width=REF_WIDTH)
pch = Device(PCH_LUT, device_type="pch", ref_width=REF_WIDTH)
LCH = sorted(float(l) for l in nch.lut["lch"].unique())
print("channel lengths in the LUT [um]:", [round(l * 1e6, 2) for l in LCH])

channel lengths in the LUT [um]: [0.13, 0.18, 0.25, 0.5, 1.0, 2.0]


## 1. Sizing with novaad

A design vector is `x = [gmid_in, gmid_load, gmid_tail, iL_in, iL_load, iL_tail, i_tail, c_cm]`, where `iL_*` index into the LUT channel lengths. Bias voltages depend on each other, so a short fixed-point iteration resolves them:

* **Load**: in the reset phase its gate sits at the CM reset level, which is designed to match the output CM, so `Vsd = Vsg`. Size it, read `Vsg`, and repeat. Then `VOCM = VDD − Vsg`.
* **Input pair**: the source sits at `Vs = VIN_CM − Vgs1` with the bulk grounded, so `Vsb = Vs` and `Vds = VOCM − Vs`.
* **Tail**: `Vds = Vs`, carrying `I_TAIL`.
* **Reset switch**: fixed sizes (`W_SWITCH_N`, `W_SWITCH_P`, `L_SWITCH` = 130 nm minimum length), with dummies at half width.


In [3]:
def size(x) -> dict:
    gmid_in, gmid_load, gmid_tail = x[:3]
    l_in, l_load, l_tail = (LCH[int(i)] for i in x[3:6])
    i_tail, c_cm = x[6], x[7]

    vsd = VDD / 2
    for _ in range(2):
        dcop, s_load = pch.sizing(Spec(vds=vsd, vsb=0.0, lch=l_load, gmoverid=gmid_load, ids=i_tail / 2), return_dcop=True)
        vsd = dcop.vgs[0]
    vocm = VDD - vsd

    vs = VIN_CM - 0.4
    for _ in range(2):
        dcop, s_in = nch.sizing(Spec(vds=vocm - vs, vsb=vs, lch=l_in, gmoverid=gmid_in, ids=i_tail / 2), return_dcop=True)
        vs = VIN_CM - dcop.vgs[0]

    _, s_tail = nch.sizing(Spec(vds=vs, vsb=0.0, lch=l_tail, gmoverid=gmid_tail, ids=i_tail), return_dcop=True)
    return dict(w_in=s_in.wch[0], l_in=l_in, w_load=s_load.wch[0], l_load=l_load,
                w_tail=s_tail.wch[0], l_tail=l_tail, i_tail=i_tail, c_cm=c_cm,
                vocm=vocm,   # output CM novaad expects: the static testbench servos avcm to hold it
                w_switch_n=W_SWITCH_N, w_switch_p=W_SWITCH_P, l_switch=L_SWITCH,
                w_switch_n_dummy=W_SWITCH_N / 2, w_switch_p_dummy=W_SWITCH_P / 2)


def area(d: dict) -> float:
    fets = 2 * d["w_in"] * d["l_in"] + 2 * d["w_load"] * d["l_load"] + 2 * d["w_tail"] * d["l_tail"]
    switches = 2 * (d["w_switch_n"] + d["w_switch_p"]) * d["l_switch"]   # main switches + half-width dummies
    return fets + switches + 2 * d["c_cm"] / CAP_DENSITY


X_EXAMPLE = [12, 8, 8, 3, 3, 3, 150e-6, 200e-15]
size(X_EXAMPLE)


{'w_in': 1.2690451302829269e-05,
 'l_in': 5e-07,
 'w_load': 1.803500298304961e-05,
 'l_load': 5e-07,
 'w_tail': 9.632611336683857e-06,
 'l_tail': 5e-07,
 'i_tail': 0.00015,
 'c_cm': 2e-13,
 'vocm': 0.6125,
 'w_switch_n': 1e-06,
 'w_switch_p': 2e-06,
 'l_switch': 1.3e-07,
 'w_switch_n_dummy': 5e-07,
 'w_switch_p_dummy': 1e-06}

## 2. Testbench: OP + AC + noise in one ngspice run

`dut(d)` writes the amplifier as a `.subckt` macro with pins `avdd avss inp inn outp outn avcm ibias clk clkb`. `testbench(d, control, vd_source, clocked)` instantiates it as `XDUT`, so its internal devices and nodes are addressed hierarchically: `spice.mos("xdut.xm1", ...)` and `v(xdut.tail)`.

* **Static reset phase** (`clocked=False`, used by the optimizer): `ph = VDD` and `phb = 0`, so the switch holds `cmfb` at `avcm` and a DC operating point exists. With `cmfb` fixed, the output common mode is open-loop: a few µA of mismatch between the loads and the tail mirror rails it. A high-gain servo (`BAVCM`) therefore sets `avcm` so the output CM sits where novaad sized it, as a replica-bias reference generator would. It acts on the common mode only, so the differential gain, AC and noise are unaffected. `simulate` returns the level it found as `r["avcm"]`.
* **Clocked** (`clocked=True`, section 6): `ph`/`phb` are non-overlapping anti-phase pulses at `F_CLK`, with gaps `T_NOV`, and `avcm` is a fixed source at that level.

The differential input comes from two E-sources around `VIN_CM`, driven by a single source `Vd`. That makes `Vd` the input of the AC transfer and the reference source of the noise analysis. `meas` finds the DC gain `A0` and the −3 dB corner `f3` of `v(outp) − v(outn)`. The noise analysis is then integrated up to `GBW = A0·f3`.

GBW is deliberately **not** read at the 0 dB crossing. Large input devices couple the input to the outputs through `Cgd`, which flattens `|H|` just above 0 dB at high frequency. The 0 dB crossing then lands far beyond the real gain-bandwidth product, and the optimizer learns to exploit that. `set sqrnoise` makes ngspice report noise in V² and V²/Hz.

`testbench` can swap the measurement block for any other analysis and drive `Vd` with any waveform. Section 6 uses this for the DC table, the Bode plots and the transients.


In [4]:
def fet(name, d, g, s, b, model, w, l):
    ng = max(1, ceil(w / REF_WIDTH - 1e-9))   # REF_WIDTH-wide fingers, like the LUT
    return f"X{name} {d} {g} {s} {b} {model} w={w:.6g} l={l:.6g} ng={ng}"


# devices checked for saturation, by their hierarchical instance name
SAT = {"xdut.xm1": spice.NMOS, "xdut.xm3": spice.PMOS, "xdut.xm5": spice.NMOS}


def measure() -> str:
    # control block used by the optimizer: OP + AC (finds GBW) + noise integrated up to GBW
    sat = " ".join(spice.mos(m, model, "vds", "vdss") for m, model in SAT.items())
    return f'''set sqrnoise
op
wrdata op.txt v(outp) v(outn) v(xdut.tail) v(avcm) i(vdd) {sat}
ac dec 20 1 100G
let gdb = db(v(outp) - v(outn))
meas ac a0 find gdb at=1
let a3 = a0 - 3
meas ac f3 when gdb=$&a3
let gbw = f3 * 10^(a0 / 20)
wrdata ac.txt gdb
noise v(outp,outn) Vd dec 20 {NOISE_FMIN} $&gbw
setplot noise1
wrdata noise_spec.txt inoise_spectrum onoise_spectrum
setplot noise2
wrdata noise_tot.txt onoise_total'''


def dut(d: dict, name: str = "fd_amp") -> str:
    # pins: supplies, differential input, differential output, CM reset level, bias current, clocks
    nmos, pmos = spice.NMOS, spice.PMOS
    return f'''.subckt {name} avdd avss inp inn outp outn avcm ibias clk clkb
* fully-differential pair, PMOS active loads, switched-capacitor CM sensing, mirrored tail
{fet("M6", "ibias", "ibias", "avss", "avss", nmos, d["w_tail"], d["l_tail"])}
{fet("M5", "tail", "ibias", "avss", "avss", nmos, d["w_tail"], d["l_tail"])}
{fet("M1", "outn", "inp", "tail", "avss", nmos, d["w_in"], d["l_in"])}
{fet("M2", "outp", "inn", "tail", "avss", nmos, d["w_in"], d["l_in"])}
{fet("M3", "outn", "cmfb", "avdd", "avdd", pmos, d["w_load"], d["l_load"])}
{fet("M4", "outp", "cmfb", "avdd", "avdd", pmos, d["w_load"], d["l_load"])}
C1 outp cmfb {d["c_cm"]}
C2 outn cmfb {d["c_cm"]}
* transmission gate resetting the common-mode node, with half-width dummies for charge-injection cancellation
{fet("MSPDL", "cmfb", "clk", "cmfb", "avdd", pmos, d["w_switch_p_dummy"], d["l_switch"])}
{fet("MSP", "cmfb", "clkb", "avcm", "avdd", pmos, d["w_switch_p"], d["l_switch"])}
{fet("MSPDR", "avcm", "clk", "avcm", "avdd", pmos, d["w_switch_p_dummy"], d["l_switch"])}
{fet("MSNDL", "cmfb", "clkb", "cmfb", "avss", nmos, d["w_switch_n_dummy"], d["l_switch"])}
{fet("MSN", "cmfb", "clk", "avcm", "avss", nmos, d["w_switch_n"], d["l_switch"])}
{fet("MSNDR", "avcm", "clkb", "avcm", "avss", nmos, d["w_switch_n_dummy"], d["l_switch"])}
.ends {name}'''


def clocks(clocked: bool) -> str:
    if not clocked:   # static reset phase: the switch holds cmfb at avcm, so a DC operating point exists
        return f"VPH ph 0 {VDD}\nVPHB phb 0 0"
    period = 1 / F_CLK
    width = period / 2 - T_NOV - 2 * T_EDGE   # each phase is high for half a period minus the gap and edges
    return (f"VPH ph 0 pulse(0 {VDD} 0 {T_EDGE} {T_EDGE} {width} {period})\n"
            f"VPHB phb 0 pulse(0 {VDD} {period / 2} {T_EDGE} {T_EDGE} {width} {period})")


def reset_level(d: dict, clocked: bool) -> str:
    if clocked:   # the level the static servo found (`simulate` returns it as r["avcm"])
        return f"VAVCM avcm 0 {d['avcm']}"
    # static: a high-gain servo, like a replica-bias generator, sets avcm so the output CM sits at the
    # novaad prediction; it acts on the common mode only, so differential gain and noise are unaffected
    vocm = d["vocm"]
    return f"BAVCM avcm 0 V={vocm} + 100 * ((v(outp) + v(outn)) / 2 - {vocm})"


# `control` replaces the measurement block; `vd_source` sets the differential input waveform;
# `clocked` drives ph/phb with the 50 MHz clocks instead of holding the reset phase
def testbench(d: dict, control: str | None = None, vd_source: str = "dc 0 ac 1", clocked: bool = False) -> str:
    return spice.header() + dut(d) + f'''
VDD vdd 0 {VDD}
IREF vdd nb {d["i_tail"]}
{reset_level(d, clocked)}
{clocks(clocked)}
VCM cmi 0 {VIN_CM}
Vd vd 0 {vd_source}
Ep inp cmi vd 0 0.5
En cmi inn vd 0 0.5
XDUT vdd 0 inp inn outp outn avcm nb ph phb fd_amp
CLP outp 0 {CL}
CLN outn 0 {CL}
.control
set wr_singlescale
set wr_vecnames
{control or measure()}
.endc
.end
'''


def simulate(d: dict, keep: bool = False) -> dict:
    wd = spice.run(testbench(d))
    op, ac = spice.read(wd / "op.txt"), spice.read(wd / "ac.txt")
    vn2 = spice.read(wd / "noise_tot.txt")["onoise_total"][0]
    gain_db, _, gbw = spice.bandwidth(ac["frequency"], ac["gdb"])
    vocm = (op["v(outp)"][0] + op["v(outn)"][0]) / 2
    # peak differential swing: each output may move until M3 (top) or M1 (bottom) leaves saturation
    swing = 2 * min(VDD - vocm - abs(op["xdut.xm3.vdss"][0]), vocm - op["v(xdut.tail)"][0] - op["xdut.xm1.vdss"][0])
    power = VDD * abs(op["i(vdd)"][0])
    snr = 10 * log10(swing ** 2 / 2 / vn2) if swing > 0 else -100.0
    r = dict(gain_db=gain_db, gbw=gbw, vocm=vocm, avcm=op["v(avcm)"][0], power=power, vn2=vn2, swing=swing, snr=snr,
             vn_in=float(np.sqrt(vn2)) / 10 ** (gain_db / 20),   # input-referred noise [Vrms]
             fom=snr + 10 * log10(gbw / power), area=area(d),
             sat=min(abs(op[f"{m}.vds"][0]) - abs(op[f"{m}.vdss"][0]) for m in SAT))
    if keep:
        r["dir"] = wd
    else:
        shutil.rmtree(wd)
    return r


simulate(size(X_EXAMPLE))


{'gain_db': 24.889377,
 'gbw': np.float64(183826269.4127355),
 'vocm': np.float64(0.612670368),
 'avcm': np.float64(0.629536827),
 'power': np.float64(0.0003358958976),
 'vn2': np.float64(1.02926137e-06),
 'swing': np.float64(0.5150341199999999),
 'snr': 51.101163333977496,
 'vn_in': 5.7782188486761914e-05,
 'fom': 168.48319207371594,
 'area': 3.078047322892294e-10,
 'sat': np.float64(-0.014075813999999992)}

## 3. The optimization problem

pymoo minimizes, so the objectives are `[spec error %, area µm², power µW, −FoM_S dB]`.

* **Spec error:** the RMS of the relative errors.
  * Targets (`SPEC_TARGETS`) are two-sided: `e_k = measured_k / target_k − 1`.
  * Limits (`SPEC_LIMITS`) are one-sided: `e_k = max(0, measured_k / limit_k − 1)`, so beating a limit costs nothing.
  * A design that hits every target and respects every limit scores 0 %.
* **No hard constraints:** every simulated design competes on the four objectives alone.

A candidate that novaad can't size (target outside the LUT) or that ngspice can't solve gets the worst objectives, so R-NSGA-II discards it.

In [5]:
XL = [GMID_IN_BOUNDS[0], GMID_LOAD_BOUNDS[0], GMID_TAIL_BOUNDS[0], 0, 0, 0, I_TAIL_BOUNDS[0], C_CM_BOUNDS[0]]
XU = [GMID_IN_BOUNDS[1], GMID_LOAD_BOUNDS[1], GMID_TAIL_BOUNDS[1]] + [len(LCH) - 1e-6] * 3 + [I_TAIL_BOUNDS[1], C_CM_BOUNDS[1]]
# display name and scale of each target, for the tables
SPEC_UNITS = {"gain_db": ("Differential gain [dB]", 1), "gbw": ("GBW [MHz]", 1e-6),
              "vocm": ("Output CM [V]", 1), "vn_in": ("Input-referred noise [uVrms]", 1e6),
              "power": ("Power dissipation [uW]", 1e6)}
FAILED = dict(F=[1e3, 1e6, 1e6, 1e3])


def spec_errors(r) -> dict:
    # relative error of each measured parameter: two-sided for targets, excess only for limits
    errors = {k: r[k] / target - 1 for k, target in SPEC_TARGETS.items()}
    errors.update({k: max(0.0, r[k] / limit - 1) for k, limit in SPEC_LIMITS.items()})
    return errors


def spec_error(r) -> float:
    # combined objective: RMS of the relative errors
    return float(np.sqrt(np.mean(np.square(list(spec_errors(r).values())))))


class FullyDifferentialAmp(ElementwiseProblem):
    def __init__(self, **kwargs):
        super().__init__(n_var=8, n_obj=4, xl=XL, xu=XU, **kwargs)

    def _evaluate(self, x, out, *args, **kwargs):
        try:
            r = simulate(size(x))
        except Exception:
            out.update(FAILED)
            return
        out["F"] = [spec_error(r) * 100, r["area"] * 1e12, r["power"] * 1e6, -r["fom"]]   # %, um^2, uW, -dB

## 4. Run R-NSGA-II with a live Pareto front

Three pieces run concurrently:
- **Optimizer thread:** `minimize(...)` evaluates each generation through a `ThreadPool`. ngspice runs as subprocesses, so threads give real parallelism.
- **Callback:** after every generation it copies the current non-dominated set into a shared `state` dict, guarded by a `Lock`.
- **Plot thread:** every `PLOT_PERIOD` seconds it reads `state`, renders the front as a self-contained Bokeh page inside an `<iframe>`, and replaces the figure below through an IPython display handle (`display_id` / `update`).

This works in Jupyter, JupyterLab and VS Code. Bokeh's own `push_notebook` needs a Jupyter comm channel that VS Code doesn't provide. The red crosses are the `REF_POINTS`.

This cell returns immediately and the figure keeps updating. The next cell waits for the optimizer to finish.

In [6]:
state = {"gen": 0, "F": np.empty((0, 4)), "best_err": float("nan"), "res": None, "error": None}
lock = Lock()
done = Event()


class Snapshot(Callback):
    def notify(self, algorithm):
        with lock:
            state["gen"] = algorithm.n_gen
            state["F"] = algorithm.opt.get("F").copy()
            state["best_err"] = float(algorithm.pop.get("F")[:, 0].min())


def optimize():
    try:
        with ThreadPool(N_WORKERS) as pool:
            problem = FullyDifferentialAmp(elementwise_runner=StarmapParallelization(pool.starmap))
            algorithm = RNSGA2(ref_points=np.array(REF_POINTS, dtype=float), epsilon=EPSILON, pop_size=POP_SIZE)
            state["res"] = minimize(problem, algorithm, ("n_gen", N_GEN), seed=SEED, callback=Snapshot(), verbose=False)
    except Exception as e:
        state["error"] = e
    finally:
        done.set()


def front_plot(F, title: str, closest=None):
    """Area vs power colored by FoM, and FoM vs spec error. F rows: [spec error %, area, power, -FoM]."""
    F = np.atleast_2d(F)
    data = dict(err=F[:, 0], area=F[:, 1], power=F[:, 2], fom=-F[:, 3])
    refs = dict(err=[p[0] for p in REF_POINTS], area=[p[1] for p in REF_POINTS],
                power=[p[2] for p in REF_POINTS], fom=[-p[3] for p in REF_POINTS])
    cmap = LinearColorMapper(palette=Viridis256, low=float(data["fom"].min()), high=float(data["fom"].max()))
    tips = [("spec error", "@err{0.0} %"), ("FoM", "@fom{0.0} dB"), ("power", "@power{0} uW"), ("area", "@area{0} um²")]
    p_ap = figure(title=title, x_axis_label="power [uW]", y_axis_label="area [um²]",
                  x_axis_type="log", y_axis_type="log", width=500, height=380, tooltips=tips)
    p_fp = figure(title="FoM vs spec error", x_axis_label="spec error, RMS [%]", y_axis_label="FoM_S [dB]",
                  width=420, height=380, tooltips=tips)
    for p, (x, y) in ((p_ap, ("power", "area")), (p_fp, ("err", "fom"))):
        p.scatter(x, y, source=ColumnDataSource(data), size=8, color={"field": "fom", "transform": cmap})
        p.scatter(x, y, source=ColumnDataSource(refs), marker="x", size=14, line_width=3, color="red")
        if closest is not None:
            c = np.atleast_2d(closest)
            near = dict(err=c[:, 0], area=c[:, 1], power=c[:, 2], fom=-c[:, 3])
            p.scatter(x, y, source=ColumnDataSource(near), size=14, fill_alpha=0, line_color="red", line_width=2)
    p_ap.add_layout(ColorBar(color_mapper=cmap, title="FoM_S [dB]"), "right")
    return row(p_ap, p_fp)


def as_iframe(layout, title: str) -> HTML:
    # a standalone page (BokehJS inlined) in an iframe renders the same in Jupyter, JupyterLab and VS Code
    page = file_html(layout, INLINE, title)
    return HTML(f'<iframe srcdoc="{escape(page)}" style="width:1000px; height:410px; border:0"></iframe>')


handle = display(HTML("<i>Pareto front: waiting for generation 1…</i>"), display_id=True)


def live_plot():
    last_gen = -1
    while True:
        finished = done.wait(PLOT_PERIOD)
        with lock:
            gen, F, best_err = state["gen"], state["F"], state["best_err"]
        if gen != last_gen and len(F):
            title = f"Generation {gen}/{N_GEN}: {len(F)} non-dominated, best spec error {best_err:.1f} %"
            handle.update(as_iframe(front_plot(F, title), title))
            last_gen = gen
        if finished:
            return


opt_thread = Thread(target=optimize, name="rnsga2", daemon=True)
plot_thread = Thread(target=live_plot, name="live-plot", daemon=True)
opt_thread.start()
plot_thread.start()

In [7]:
# wait for the optimizer (and the last plot refresh) before analysing the result
opt_thread.join()
plot_thread.join()
if state["error"]:
    raise state["error"]
res = state["res"]
print(f"{res.algorithm.evaluator.n_eval} evaluations in {res.exec_time:.1f} s, "
      f"{len(res.F)} non-dominated designs, best spec error {res.pop.get('F')[:, 0].min():.2f} %")

1920 evaluations in 110.6 s, 48 non-dominated designs, best spec error 4.17 %


## 5. Closest designs

The final Pareto front (the live figure above keeps its last frame), followed by the `N_CLOSEST` designs closest to the spec targets, ranked by spec error (ties broken by FoM). The closest designs are circled in red on the front.

* **Where they come from:** the whole final population, not only the non-dominated front. A design can be close to the targets and still be dominated on area or power.
* **Error columns:** each `err` column is one spec's relative error, two-sided for targets and excess-only for limits.
* **`sat margin`:** the smallest `|Vds| − |Vdss|` among the transistors (`vdss` is PSP103's saturation voltage). A negative value means a device is out of saturation.

In [8]:
X = res.pop.get("X")
rows = []
for i, x in enumerate(X):
    try:
        rows.append({"i": i, **size(x), **simulate(size(x))})
    except Exception:
        pass   # novaad could not size it or ngspice failed: it can't be a close design anyway
designs = DataFrame(rows).set_index("i")
designs["spec_error"] = designs.apply(spec_error, axis=1)
table = DataFrame({
    "spec error [%]": designs["spec_error"] * 100,
    **{f"err {SPEC_UNITS.get(k, (k, 1))[0].split(' [')[0]} [%]": designs.apply(lambda r: spec_errors(r)[k], axis=1) * 100
       for k in {**SPEC_TARGETS, **SPEC_LIMITS}},
    "FoM_S [dB]": designs["fom"],
    "power [uW]": designs["power"] * 1e6,
    "area [um²]": designs["area"] * 1e12,
    "gain [dB]": designs["gain_db"],
    "GBW [MHz]": designs["gbw"] / 1e6,
    "VOCM [V]": designs["vocm"],
    "noise in [uVrms]": designs["vn_in"] * 1e6,
    "SNR [dB]": designs["snr"],
    "sat margin [V]": designs["sat"],
    "gm/id in": X[designs.index, 0], "gm/id load": X[designs.index, 1], "gm/id tail": X[designs.index, 2],
    "L in [um]": designs["l_in"] * 1e6, "L load [um]": designs["l_load"] * 1e6, "L tail [um]": designs["l_tail"] * 1e6,
    "W in [um]": designs["w_in"] * 1e6, "W load [um]": designs["w_load"] * 1e6, "W tail [um]": designs["w_tail"] * 1e6,
    "I tail [uA]": designs["i_tail"] * 1e6, "C_CM [fF]": designs["c_cm"] * 1e15,
}, index=designs.index).sort_values(["spec error [%]", "FoM_S [dB]"], ascending=[True, False])
# keep distinct designs only: neighbours that round to the same spec error, power and area are duplicates
key = DataFrame({"err": table["spec error [%]"].round(2), "p": table["power [uW]"].round(0), "a": table["area [um²]"].round(0)})
table = table[~key.duplicated()]
order = table.index[:N_CLOSEST]   # rows of X, closest first
table = table.loc[order].reset_index(drop=True)

closest = np.column_stack([table["spec error [%]"], table["area [um²]"], table["power [uW]"], -table["FoM_S [dB]"]])
show(front_plot(res.F, f"Final Pareto front: {len(res.F)} designs, {len(table)} closest circled", closest))
table.round(2)

,spec error [%],err Differential gain [%],err GBW [%],err Output CM [%],err Power dissipation [%],err Input-referred noise [%],FoM_S [dB],power [uW],area [um²],gain [dB],...,gm/id load,gm/id tail,L in [um],L load [um],L tail [um],W in [um],W load [um],W tail [um],I tail [uA],C_CM [fF]
0,4.17,0.22,0.13,2.09,-9.09,0.0,171.64,454.57,496.23,26.06,...,8.12,15.92,1.0,0.50,0.5,182.35,22.84,71.16,189.95,27.57
1,4.22,0.16,2.13,2.08,-8.95,0.0,171.78,455.26,613.16,26.04,...,8.25,15.53,1.0,0.50,0.5,241.60,22.78,70.96,189.43,26.57
2,4.35,0.44,0.64,4.17,-8.76,0.0,172.33,456.22,630.80,26.11,...,8.57,15.33,1.0,0.50,0.5,242.06,25.46,71.10,189.79,37.01
3,4.91,-0.06,-1.38,0.01,-10.89,0.0,170.98,445.56,441.35,25.98,...,7.88,12.04,1.0,0.50,0.5,181.85,20.75,30.05,189.43,19.56
4,5.13,0.20,-2.92,2.09,-10.88,0.0,171.55,445.59,455.27,26.05,...,8.27,12.37,1.0,0.50,0.5,181.83,22.77,30.05,189.42,28.50
5,5.16,0.18,-1.15,2.08,-11.30,0.0,171.73,443.51,611.52,26.05,...,8.28,15.30,1.0,0.50,0.5,235.11,22.16,69.06,184.34,36.98
6,5.25,0.41,-0.73,4.17,-10.94,0.0,172.27,445.28,567.97,26.11,...,8.60,11.93,1.0,0.50,0.5,240.22,25.26,29.47,188.34,24.02
7,5.26,-2.86,-3.24,0.01,-10.94,0.0,169.13,445.32,449.41,25.26,...,8.46,12.27,1.0,0.25,0.5,181.75,12.76,30.03,189.33,36.54
8,5.31,0.18,2.59,2.08,-11.41,0.0,171.77,442.97,576.46,26.05,...,8.28,15.33,1.0,0.50,0.5,234.82,22.14,68.97,184.12,11.19
9,5.58,0.26,-0.89,2.09,-12.26,0.0,171.55,438.69,430.73,26.07,...,8.21,15.13,1.0,0.50,0.5,168.13,22.13,57.09,184.02,10.85


## 6. Inspect one design

Pick a row of the table above (`PICK = 0` is the design closest to the spec targets).

### 6.1 DC operating point

One `op` run saves every node voltage and element current. `utils.dc_tables` maps each element's terminals to nets through InSpice's parse of the netlist, with the `XDUT` instance flattened.

In [9]:
PICK = 0

x = X[order[PICK]]
d = size(x)
bench = testbench(d)
op = spice.read(spice.run(testbench(d, f"op\nwrdata op.txt {dc_vectors(bench)}")) / "op.txt")
devices, nodes = dc_tables(bench, op)
d["avcm"] = op["v(avcm)"][0]   # CM reset level found by the servo: a fixed source in the clocked runs
display(devices.round(4).fillna(""))
nodes.round(4)

,type,nets,I [uA],V(+) [V],V(-) [V],V(c+) [V],V(c-) [V],V(d) [V],V(g) [V],V(s) [V],V(b) [V],vgs [V],vds [V],vsb [V],vth [V],vdss [V],gm [uS],gds [uS],region
element,,,,,,,,,,,,,,,,,,,
VDD,V,+=vdd -=0,-378.8115,1.2,0.0,,,,,,,,,,,,,,
IREF,I,+=vdd -=nb,189.9524,1.2,0.3278,,,,,,,,,,,,,,
BAVCM,B,+=avcm -=0,,0.6137,0.0,,,,,,,,,,,,,,
VPH,V,+=ph -=0,-0.0,1.2,0.0,,,,,,,,,,,,,,
VPHB,V,+=phb -=0,0.0,0.0,0.0,,,,,,,,,,,,,,
VCM,V,+=cmi -=0,-0.0,0.6,0.0,,,,,,,,,,,,,,
Vd,V,+=vd -=0,0.0,0.0,0.0,,,,,,,,,,,,,,
Ep,E,+=inp -=cmi c+=vd c-=0,-0.0,0.6,0.6,0.0,0.0,,,,,,,,,,,,
En,E,+=cmi -=inn c+=vd c-=0,0.0,0.6,0.6,0.0,0.0,,,,,,,,,,,,


,V [V]
node,
avcm,0.6137
cmi,0.6000
inn,0.6000
inp,0.6000
nb,0.3278
outn,0.6125
outp,0.6125
ph,1.2000
phb,0.0000


### 6.2 AC in the reset phase (static): Bode plot of `(v(outp) − v(outn)) / vd`

A linear AC analysis needs a DC operating point, so it runs with the switch held closed (`ph = VDD`). It is the reference for the clocked response below.


In [10]:
AC = '''ac dec 50 1 100G
let vod = v(outp) - v(outn)
let gdb = db(vod)
let gph = 180 / pi * cph(vod)
wrdata ac.txt gdb gph'''
ac = spice.read(spice.run(testbench(d, AC)) / "ac.txt")
f, mag, phase = ac["frequency"].to_numpy(), ac["gdb"].to_numpy(), ac["gph"].to_numpy()
gain_db, f3, gbw = spice.bandwidth(f, mag)
print(f"DC gain {gain_db:.2f} dB, f-3dB {f3 / 1e6:.3f} MHz, GBW = A0·f-3dB {gbw / 1e6:.1f} MHz, "
      f"|H| at GBW {np.interp(gbw, f, mag):.1f} dB")

p_mag = figure(title="Magnitude", x_axis_type="log", x_axis_label="f [Hz]", y_axis_label="|vod/vd| [dB]", width=900, height=280)
p_mag.line(f, mag, line_width=2)
p_mag.scatter([f3], [gain_db - 3], size=8, color="red")
p_mag.line([f3, gbw], [gain_db, 0], line_dash="dashed", color="gray", legend_label="single-pole asymptote to GBW")
p_ph = figure(title="Phase", x_axis_type="log", x_axis_label="f [Hz]", y_axis_label="phase [deg]", width=900, height=280, x_range=p_mag.x_range)
p_ph.line(f, phase, line_width=2)
show(gridplot([[p_mag], [p_ph]]))

DC gain 26.06 dB, f-3dB 24.931 MHz, GBW = A0·f-3dB 500.8 MHz, |H| at GBW 0.7 dB


### 6.3 Periodic steady state: swept-sine Bode of the clocked amplifier

ngspice has no PAC analysis (its `pss` only handles autonomous oscillators), so the periodic small-signal transfer is measured directly in the time domain. This is the baseband (sideband 0) response that PSS + PAC would give:

1. Each test frequency is snapped to `f = (m/N)·F_CLK` with `N ≤ PSS_MAX_CYCLES`. The input sine and the clock then share the common period `T = N/F_CLK`, so the steady state is periodic in `T`.
2. A clocked transient runs for an integer number of periods `T` to settle: at least `10·τ` and `PSS_SETTLE_CYCLES` clock cycles. Then one more period `T` is recorded and `linearize`d onto a uniform `1/(F_CLK·STEPS_PER_CLK)` grid.
3. A DFT at `f` of `vod` and of `vd` over that period gives `H(f) = Vod(f) / Vd(f)`, magnitude and phase. Clock feedthrough and the harmonics fall on other bins.

The points run in parallel. The grey curve is the static AC from 6.2, and the dashed red line marks the Nyquist frequency `F_CLK/2`.


In [11]:
tau = 1 / (2 * pi * f3)   # from the static AC above


def pss_point(f_target: float) -> tuple[float, complex]:
    """(coherent test frequency, vod/vd at that frequency) in periodic steady state."""
    ratio = Fraction(f_target / F_CLK).limit_denominator(PSS_MAX_CYCLES)
    f = float(ratio) * F_CLK                  # f/F_CLK = m/N: input and clock share the period T = N/F_CLK
    period = ratio.denominator / F_CLK
    settle = ceil(max(10 * tau, PSS_SETTLE_CYCLES / F_CLK) / period) * period
    tstep = 1 / (F_CLK * STEPS_PER_CLK)
    control = f"""tran {tstep} {settle + period} {settle} {tstep}
let vod = v(outp) - v(outn)
linearize vod v(vd)
wrdata pss.txt vod v(vd)"""
    wd = spice.run(testbench(d, control, f"dc 0 sin(0 {PSS_AMP} {f})", clocked=True))
    data = spice.read(wd / "pss.txt").iloc[:-1]   # exactly one period: drop the repeated end point
    shutil.rmtree(wd)
    e = np.exp(-2j * pi * f * data["time"].to_numpy())   # DFT at the test frequency
    return f, complex(data["vod"].to_numpy() @ e / (data["v(vd)"].to_numpy() @ e))


with ThreadPool(N_WORKERS) as pool:
    pss = pool.map(pss_point, np.geomspace(PSS_FMIN, PSS_FMAX, PSS_POINTS))
f_pss = np.array([p[0] for p in pss])
h_pss = np.array([p[1] for p in pss])
mag_pss = 20 * np.log10(np.abs(h_pss))
phase_pss = np.degrees(np.unwrap(np.angle(h_pss)))
gain_pss_db, f3_pss, gbw_pss = spice.bandwidth(f_pss, mag_pss)
print(f"clocked: DC gain {gain_pss_db:.2f} dB, f-3dB {f3_pss / 1e6:.3f} MHz, GBW {gbw_pss / 1e6:.1f} MHz "
      f"(static AC: {gain_db:.2f} dB, {f3 / 1e6:.3f} MHz, {gbw / 1e6:.1f} MHz)")

nyquist = dict(location=F_CLK / 2, dimension="height", line_color="red", line_dash="dashed")
p_mag = figure(title=f"Magnitude, clocked at {F_CLK / 1e6:g} MHz", x_axis_type="log", x_axis_label="f [Hz]",
               y_axis_label="|vod/vd| [dB]", width=900, height=280)
p_mag.line(f, mag, color="gray", legend_label="AC, reset phase (static)")
p_mag.line(f_pss, mag_pss, line_width=2, legend_label="PSS, clocked")
p_mag.scatter(f_pss, mag_pss, size=6)
p_ph = figure(title="Phase", x_axis_type="log", x_axis_label="f [Hz]", y_axis_label="phase [deg]",
              width=900, height=280, x_range=p_mag.x_range)
p_ph.line(f, phase, color="gray")
p_ph.line(f_pss, phase_pss, line_width=2)
p_ph.scatter(f_pss, phase_pss, size=6)
for p in (p_mag, p_ph):
    p.add_layout(Span(**nyquist))
p_mag.legend.location = "bottom_left"
show(gridplot([[p_mag], [p_ph]]))


clocked: DC gain 25.99 dB, f-3dB 24.856 MHz, GBW 495.2 MHz (static AC: 26.06 dB, 24.931 MHz, 500.8 MHz)


### 6.4 Transient (clocked): sine, ramp and step

Three clocked runs (`ph`/`phb` at `F_CLK`) with different waveforms on the differential input `vd`. The time scales come from this design's `f−3dB`, with `τ = 1/(2π·f−3dB)`:
- **Sine:** at `TRAN_FREQ_RATIO·f−3dB`.
- **Ramp:** `vd` from `RAMP_LOW` to `RAMP_HIGH` in `RAMP_TAUS·τ`, so each input sweeps the full supply. The differential output traces the whole large-signal transfer curve and clips on both sides.
- **Step:** a pulse of `vd` from `STEP_LOW` to `STEP_HIGH` and back, again rail to rail on each input. The pair fully steers the tail current to one side, so the differential output **slews** at roughly `I_TAIL/CL`. The transistors' own capacitances lower it. With one input at the rail, the tail node rises and a short-channel tail mirror delivers more than `I_TAIL`, which raises it. The slew rate is measured as the slope between 40 % and 60 % of each transition (`spice.slew_rate`), which ignores the kick the input edge couples through `Cgd`.

Input and output share the same voltage axis, so the gain is visible directly as the ratio of the two swings.

In [12]:
def tran(vd_source: str, tstop: float):
    tstep = min(tstop / 2000, 1 / (F_CLK * STEPS_PER_CLK))   # resolve the clock edges
    control = f"tran {tstep} {tstop} 0 {tstep}\nlet vod = v(outp) - v(outn)\nwrdata tran.txt v(vd) vod"
    return spice.read(spice.run(testbench(d, control, vd_source, clocked=True)) / "tran.txt")


f_sine = TRAN_FREQ_RATIO * f3
sine = tran(f"dc 0 sin(0 {TRAN_AMP} {f_sine})", 6 / f_sine)
ramp = tran(f"pwl(0 {RAMP_LOW} {RAMP_TAUS * tau} {RAMP_HIGH})", RAMP_TAUS * tau)
t_win = STEP_TAUS * tau
t_up, t_down = t_win / 10, t_win / 2
t_edge = t_win / 1000   # input rise/fall time
step = tran(f"pulse({STEP_LOW} {STEP_HIGH} {t_up} {t_edge} {t_edge} {t_down - t_up - t_edge} 1)", t_win)

last = sine["time"] >= 5 / f_sine
gain_tran_db = 20 * log10((sine["vod"][last].max() - sine["vod"][last].min()) / 2 / TRAN_AMP)
sr_up = spice.slew_rate(step["time"], step["vod"], t_up, t_down)
sr_down = spice.slew_rate(step["time"], step["vod"], t_down, t_win)
sr_expected = d["i_tail"] / CL
print(f"sine gain {gain_tran_db:.2f} dB | ramp output {ramp['vod'].min():.2f} .. {ramp['vod'].max():.2f} V | "
      f"slew rate {sr_up / 1e6:.1f} / {sr_down / 1e6:.1f} V/us up/down (bound I_TAIL/CL = {sr_expected / 1e6:.1f})")

ns = dict(t_scale=1e9, t_unit="ns", in_label="vd", out_label="vod")
show(gridplot([
    [tran_plot(f"Sine: {TRAN_AMP * 1e3:g} mV @ {f_sine / 1e6:.3g} MHz", sine["time"], sine["v(vd)"], sine["vod"], **ns)],
    [tran_plot(f"Ramp: vd {RAMP_LOW:g} V to {RAMP_HIGH:g} V in {RAMP_TAUS * tau * 1e9:.0f} ns", ramp["time"], ramp["v(vd)"], ramp["vod"], **ns)],
    [tran_plot(f"Step: vd {STEP_LOW:g} V to {STEP_HIGH:g} V and back", step["time"], step["v(vd)"], step["vod"], **ns)],
]))

sine gain 25.95 dB | ramp output -1.16 .. 1.15 V | slew rate 289.9 / 176.5 V/us up/down (bound I_TAIL/CL = 379.9)


### 6.5 Clock phases and the CM reset

A few clock periods after settling, with no differential input. During `ph` the transmission gate pulls `cmfb` to `avcm`. During `phb` it opens: `cmfb` holds its charge and follows the output common mode through `C_CM`. The steps at the clock edges are the residual charge injection and clock feedthrough the dummies don't cancel.


In [13]:
N_SHOW = 4   # clock periods to plot
control = f"""tran {1 / (F_CLK * STEPS_PER_CLK)} {(PSS_SETTLE_CYCLES + N_SHOW) / F_CLK} {PSS_SETTLE_CYCLES / F_CLK}
let vocm = (v(outp) + v(outn)) / 2
wrdata clk.txt v(ph) v(phb) v(xdut.cmfb) v(avcm) vocm"""
clk = spice.read(spice.run(testbench(d, control, "dc 0", clocked=True)) / "clk.txt")
t_ns = (clk["time"] - clk["time"].iloc[0]) * 1e9
print(f"cmfb {clk['v(xdut.cmfb)'].min():.3f} .. {clk['v(xdut.cmfb)'].max():.3f} V (avcm {d['avcm']:.3f} V), "
      f"output CM {clk['vocm'].min():.3f} .. {clk['vocm'].max():.3f} V")

p_clk = figure(title="Clocks", x_axis_label="t [ns]", y_axis_label="V", width=900, height=200)
p_clk.line(t_ns, clk["v(ph)"], line_width=2, legend_label="ph (reset)")
p_clk.line(t_ns, clk["v(phb)"], line_width=2, color="orange", legend_label="phb (amplify)")
p_cm = figure(title="Common mode", x_axis_label="t [ns]", y_axis_label="V", width=900, height=260, x_range=p_clk.x_range)
p_cm.line(t_ns, clk["v(xdut.cmfb)"], line_width=2, legend_label="cmfb")
p_cm.line(t_ns, clk["v(avcm)"], line_dash="dashed", color="gray", legend_label="avcm")
p_cm.line(t_ns, clk["vocm"], line_width=2, color="green", legend_label="output CM")
for p in (p_clk, p_cm):
    p.legend.location = "top_right"
show(gridplot([[p_clk], [p_cm]]))


cmfb 0.607 .. 0.616 V (avcm 0.614 V), output CM 0.598 .. 0.605 V


### 6.6 Noise and spec check

The design is re-simulated with the optimizer's own measurement (`simulate`), and the files are kept so the output noise density can be plotted. The final area is also given as the side of the equivalent square footprint (`√A × √A`): transistor gate areas, the reset switches and the two MIM sensing capacitors, before layout overheads.

In [14]:
r = simulate(d, keep=True)
noise = spice.read(r["dir"] / "noise_spec.txt")
p = figure(title="Noise density", x_axis_type="log", y_axis_type="log", x_axis_label="f [Hz]",
           y_axis_label="nV/√Hz", width=900, height=300)
p.line(noise["frequency"], np.sqrt(noise["inoise_spectrum"]) * 1e9, line_width=2, legend_label="input-referred")
p.line(noise["frequency"], np.sqrt(noise["onoise_spectrum"]) * 1e9, line_width=2, color="orange", legend_label="output")
show(p)

errors = spec_errors(r)
report = [
    (SPEC_UNITS.get(k, (k, 1))[0], f"target {t * SPEC_UNITS.get(k, (k, 1))[1]:g}",
     f"{r[k] * SPEC_UNITS.get(k, (k, 1))[1]:.2f} ({errors[k]:+.1%})")
    for k, t in SPEC_TARGETS.items()
] + [
    (SPEC_UNITS.get(k, (k, 1))[0], f"limit <= {t * SPEC_UNITS.get(k, (k, 1))[1]:g}",
     f"{r[k] * SPEC_UNITS.get(k, (k, 1))[1]:.2f} (excess {errors[k]:.1%})")
    for k, t in SPEC_LIMITS.items()
] + [
    ("Spec error, RMS [%]", "minimize", f"{spec_error(r) * 100:.2f}"),
    # the sine sits on the pole's roll-off: compare it with the AC gain at the same frequency
    ("Transient gain [dB]", f"AC gain at {f_sine / 1e6:.3g} MHz: {np.interp(f_sine, f, mag):.2f}", f"{gain_tran_db:.2f}"),
    ("Clocked DC gain, PSS [dB]", f"static AC: {gain_db:.2f}", f"{gain_pss_db:.2f}"),
    ("Clocked GBW, PSS [MHz]", f"static AC: {gbw / 1e6:.1f}", f"{gbw_pss / 1e6:.1f}"),
    ("CM reset level avcm [V]", "servo-solved in the reset phase", f"{r['avcm']:.3f}"),
    ("Slew rate, vod up/down [V/us]", f"reference I_TAIL/CL = {sr_expected / 1e6:.1f}", f"{sr_up / 1e6:.1f} / {sr_down / 1e6:.1f}"),
    ("Saturation margin [V]", "> 0: all devices saturated", f"{r['sat']:.2f}"),
    ("Diff. swing, peak [V]", "", f"{r['swing']:.2f}"),
    ("Output noise [uVrms]", "", f"{np.sqrt(r['vn2']) * 1e6:.2f}"),
    ("SNR [dB]", "", f"{r['snr']:.2f}"),
    ("Power [uW]", "minimize", f"{r['power'] * 1e6:.2f}"),
    ("Area [um²]", "minimize", f"{r['area'] * 1e12:.2f}"),
    ("Area, equivalent square", "", footprint(r["area"])),
    ("Schreier FoM [dB]", "maximize", f"{r['fom']:.2f}"),
]
shutil.rmtree(r["dir"])
DataFrame(report, columns=["metric", "target", "ngspice"]).set_index("metric")

,target,ngspice
metric,,
Differential gain [dB],target 26,26.06 (+0.2%)
GBW [MHz],target 500,500.64 (+0.1%)
Output CM [V],target 0.6,0.61 (+2.1%)
Power dissipation [uW],target 500,454.57 (-9.1%)
Input-referred noise [uVrms],limit <= 60,33.14 (excess 0.0%)
"Spec error, RMS [%]",minimize,4.17
Transient gain [dB],AC gain at 2.49 MHz: 26.02,25.95
"Clocked DC gain, PSS [dB]",static AC: 26.06,25.99
"Clocked GBW, PSS [MHz]",static AC: 500.8,495.2
